# Step 3: Random-forest selection, then regressor v classifier
Folds run forward in time: train 2021 -> 2022, ..., train 2021-24 -> 2025. **2026 is not loaded.** Settings are fixed and conservative, not tuned. Selection happens inside each fold's training seasons.

In [1]:
import os, json, warnings; warnings.filterwarnings('ignore'); ROOT=os.path.expanduser('~/hundred-momentum')
import numpy as np, pandas as pd, lightgbm as lgb
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.linear_model import RidgeCV, LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.model_selection import GroupKFold, GroupShuffleSplit
from sklearn.metrics import log_loss, roc_auc_score
feats = json.load(open(ROOT+'/data/clean/feature_list.json'))
D = pd.read_parquet(ROOT+'/data/clean/features.parquet'); D = D[(D.season <= 2025) & (D.full_match == 1)].reset_index(drop=True); assert D.season.max() == 2025
NAMES = ['W','0','1','2','3','4','5+']
def cls(d): return np.where(d.is_wicket == 1, 0, 1 + np.minimum(d.runs_total, 5))
D['cls'] = cls(D); D['value'] = np.where(D.is_wicket == 1, -6, D.runs_total)
X = D[feats].astype(float); yr = D.runs_total.values.astype(float); yc = D.cls.values; yv = D.value.values.astype(float); gid = D.match_id.values; S = D.season.values
print(D.shape, dict(zip(NAMES, np.bincount(yc))))
FOLDS = [(np.where(S < s)[0], np.where(S == s)[0], s) for s in (2022, 2023, 2024, 2025)]

(30846, 106) {'W': np.int64(1907), '0': np.int64(8026), '1': np.int64(12865), '2': np.int64(2474), '3': np.int64(171), '4': np.int64(3624), '5+': np.int64(1779)}


## 3.1 Random-forest selection (per fold, training seasons only)
Permutation importance on held-out *matches* within the training seasons, for both a regression and a classification forest. Keep a feature if shuffling it makes either forest worse on average.

In [2]:
def perm_select(tr, seed=0, reps=3):
    rng = np.random.default_rng(seed); imp_r = np.zeros(len(feats)); imp_c = np.zeros(len(feats)); Xn = X.iloc[tr].fillna(X.iloc[tr].median()).values
    for a, b in GroupKFold(3).split(Xn, groups=gid[tr]):
        kw = dict(n_estimators=150, min_samples_leaf=40, max_features=0.33, n_jobs=-1, random_state=seed)
        R = RandomForestRegressor(**kw).fit(Xn[a], yr[tr][a]); C = RandomForestClassifier(**kw).fit(Xn[a], yc[tr][a])
        lab = C.classes_; y1, y2 = yr[tr][b], yc[tr][b]; ok = np.isin(y2, lab)
        r0 = ((R.predict(Xn[b]) - y1) ** 2).mean(); c0 = log_loss(y2[ok], C.predict_proba(Xn[b][ok]), labels=lab)
        for j in range(len(feats)):
            for _ in range(reps):
                Z = Xn[b].copy(); Z[:, j] = rng.permutation(Z[:, j])
                imp_r[j] += (((R.predict(Z) - y1) ** 2).mean() - r0) / (3 * reps) / r0; imp_c[j] += (log_loss(y2[ok], C.predict_proba(Z[ok]), labels=lab) - c0) / (3 * reps) / c0
    return imp_r, imp_c
SEL, IMP = [], []
for tr, va, s in FOLDS:
    r, c = perm_select(tr); IMP.append(pd.DataFrame({'reg': r, 'clf': c}, index=feats)); SEL.append([f for f, x, y in zip(feats, r, c) if x > 0 or y > 0]); print(s, 'kept', len(SEL[-1]), 'of', len(feats))
cnt = pd.Series([f for s in SEL for f in s]).value_counts(); avg = sum(IMP) / 4
print('kept in >=3 folds:', int((cnt >= 3).sum())); print((avg.sort_values('clf', ascending=False).head(15) * 100).round(3).to_string())

2022 kept 48 of 71


2023 kept 40 of 71


2024 kept 48 of 71


2025 kept 52 of 71
kept in >=3 folds: 42
                      reg    clf
wkts_per_ball_left  0.390  0.299
balls_left          0.018  0.228
ball_no             0.062  0.194
powerplay           0.015  0.145
bat_balls           0.270  0.129
bat_runs            0.184  0.128
part_runs           0.190  0.072
score_pre           0.076  0.050
since_wicket        0.104  0.036
part_balls          0.105  0.032
req_rate            0.019  0.026
rate_gap           -0.008  0.017
batph_hist_dot      0.004  0.010
bowl_balls         -0.007  0.008
bat_hist_bnd       -0.022  0.008


## 3.2 Models
Both families see the same data, folds and settings. Classifier expected runs = sum of P(class) x mean runs of that class in training.

In [3]:
LGB = dict(n_estimators=2000, learning_rate=.02, max_depth=4, num_leaves=12, min_child_samples=100, subsample=.8, subsample_freq=1, colsample_bytree=.5, reg_lambda=5, random_state=42, n_jobs=-1, verbose=-1)
def inner(tr): a, b = next(GroupShuffleSplit(1, test_size=.2, random_state=7).split(tr, groups=gid[tr])); return tr[a], tr[b]
def lgbm(K, tr, t, fs):
    a, b = inner(tr); return K(**LGB).fit(X.iloc[a][fs], t[a], eval_set=[(X.iloc[b][fs], t[b])], callbacks=[lgb.early_stopping(60, verbose=False)])
lin = lambda est: make_pipeline(SimpleImputer(strategy='median'), StandardScaler(), est)
def full(P, lab): Q = np.full((len(P), 7), 1e-9); Q[:, lab] = P; return Q / Q.sum(1, keepdims=True)
def preds(tr, va, fs):
    A, V = X.iloc[tr][fs], X.iloc[va][fs]; med = A.median(); rf = dict(n_estimators=400, min_samples_leaf=40, max_features=.33, n_jobs=-1, random_state=1); out = {}
    for t, nm in [(yr, 'runs'), (yv, 'value')]:
        out['Reg: ridge', nm] = lin(RidgeCV(np.logspace(0, 5, 11))).fit(A, t[tr]).predict(V)
        out['Reg: random forest', nm] = RandomForestRegressor(**rf).fit(A.fillna(med), t[tr]).predict(V.fillna(med))
        out['Reg: LightGBM', nm] = lgbm(lgb.LGBMRegressor, tr, t, fs).predict(V)
    m1 = lin(LogisticRegression(C=.01, max_iter=2000)).fit(A, yc[tr]); m2 = RandomForestClassifier(**rf).fit(A.fillna(med), yc[tr]); m3 = lgbm(lgb.LGBMClassifier, tr, yc, fs)
    for nm, P in [('Clf: logistic', full(m1.predict_proba(V), m1.classes_)), ('Clf: random forest', full(m2.predict_proba(V.fillna(med)), m2.classes_)), ('Clf: LightGBM', full(m3.predict_proba(V), m3.classes_))]:
        out[nm, 'prob'] = P
        for t, k in [(yr, 'runs'), (yv, 'value')]: out[nm, k] = P @ np.array([t[tr][yc[tr] == c].mean() if (yc[tr] == c).any() else 0 for c in range(7)])
    # simple table: phase x wickets in hand
    key = lambda i: pd.Series(np.minimum(D.ball_no.values[i] - 1, 99) // 25 * 10 + np.minimum(D.wkts_pre.values[i], 6))
    for t, nm in [(yr, 'runs'), (yv, 'value')]:
        tb = pd.Series(t[tr]).groupby(key(tr)).agg(['sum','count']); mu = t[tr].mean(); out['Baseline: phase x wickets', nm] = key(va).map((tb['sum'] + 50 * mu) / (tb['count'] + 50)).fillna(mu).values
        out['Baseline: mean', nm] = np.full(len(va), mu)
    pr = np.bincount(yc[tr], minlength=7) / len(tr); out['Baseline: mean', 'prob'] = np.tile(pr, (len(va), 1))
    tb = pd.DataFrame(np.eye(7)[yc[tr]]).groupby(key(tr).values).sum(); tb = tb.add(50 * pr, axis=1); tb = tb.div(tb.sum(1), axis=0); out['Baseline: phase x wickets', 'prob'] = np.vstack([tb.loc[k] if k in tb.index else pr for k in key(va)])
    return out
def score(o, va, tr):
    rows = []; O = np.eye(7)[yc[va]]
    for m in sorted({k[0] for k in o}):
        r = dict(model=m)
        for t, nm in [(yr, 'runs'), (yv, 'value')]:
            e = t[va] - o[m, nm]; r[f'R2_{nm}'] = 1 - (e ** 2).sum() / ((t[va] - t[tr].mean()) ** 2).sum(); r[f'RMSE_{nm}'] = (e ** 2).mean() ** .5
        r['AUC_boundary'] = roc_auc_score(D.runs_bat.values[va] >= 4, o[m, 'prob'][:, 5:].sum(1) if (m, 'prob') in o else o[m, 'runs'])
        if (m, 'prob') in o: P = o[m, 'prob']; r.update(logloss=log_loss(yc[va], P, labels=range(7)), brier=((P - O) ** 2).sum(1).mean(), AUC_wicket=roc_auc_score(yc[va] == 0, P[:, 0]), AUC_dot=roc_auc_score(yc[va] == 1, P[:, 1]))
        rows.append(r)
    return pd.DataFrame(rows)
ALL, OOF = [], {}
for (tr, va, s), fs in zip(FOLDS, SEL):
    for tag, use in [('selected', fs), ('all', feats)]:
        o = preds(tr, va, use); sc = score(o, va, tr); sc['season'] = s; sc['features'] = tag; ALL.append(sc)
        if tag == 'selected': OOF[s] = (va, o)
A = pd.concat(ALL); A.to_csv(ROOT+'/data/clean/cv_results.csv', index=False); pd.set_option('display.width', 250)
for tag in ('selected','all'): print(tag.upper()); print(A[A.features == tag].groupby('model').mean(numeric_only=True).drop(columns='season').round(4).to_string())
print('Without fold 1 (selected)'); print(A[(A.features == 'selected') & (A.season > 2022)].groupby('model')[['R2_runs','R2_value','logloss','AUC_wicket']].mean().round(4).to_string())
print('R2_runs by season'); print(A[A.features == 'selected'].pivot(index='model', columns='season', values='R2_runs').round(4).to_string()); print('logloss by season'); print(A[A.features == 'selected'].pivot(index='model', columns='season', values='logloss').dropna().round(4).to_string())

SELECTED
                           R2_runs  RMSE_runs  R2_value  RMSE_value  AUC_boundary  logloss   brier  AUC_wicket  AUC_dot
model                                                                                                                  
Baseline: mean              0.0000     1.6594    0.0000      2.4326        0.5000   1.5405  0.7333      0.5000   0.5000
Baseline: phase x wickets   0.0089     1.6521    0.0016      2.4306        0.5704   1.5138  0.7211      0.5750   0.6076
Clf: LightGBM               0.0281     1.6359    0.0052      2.4262        0.6120   1.4995  0.7166      0.5978   0.6336
Clf: logistic               0.0185     1.6439   -0.0001      2.4326        0.6024   1.5081  0.7191      0.5952   0.6293
Clf: random forest          0.0283     1.6357    0.0065      2.4247        0.6131   1.4969  0.7159      0.6029   0.6333
Reg: LightGBM               0.0280     1.6360    0.0046      2.4270        0.6008      NaN     NaN         NaN      NaN
Reg: random forest          0.0

## 3.3 Is the difference real? Match-level bootstrap on pooled validation balls

In [4]:
va = np.concatenate([OOF[s][0] for s in OOF]); cat = lambda m, k: np.concatenate([OOF[s][1][m, k] for s in OOF]); mg = gid[va]; um = np.unique(mg); pos = {m: np.where(mg == m)[0] for m in um}; rng = np.random.default_rng(1); BS = [np.concatenate([pos[m] for m in rng.choice(um, len(um))]) for _ in range(1000)]
def ci(d, nm): b = [d[i].mean() for i in BS]; print(f'{nm}: {d.mean():+.5f}  95% [{np.percentile(b,2.5):+.5f}, {np.percentile(b,97.5):+.5f}]')
se = lambda m, k, t: (t[va] - cat(m, k)) ** 2; ll = lambda m: -np.log(np.clip(cat(m, 'prob')[np.arange(len(va)), yc[va]], 1e-12, 1))
print('negative = first model better'); 
for a, b in [('Clf: LightGBM','Reg: LightGBM'), ('Clf: logistic','Reg: ridge'), ('Clf: random forest','Reg: random forest'), ('Clf: LightGBM','Clf: logistic'), ('Reg: LightGBM','Reg: ridge'), ('Clf: LightGBM','Baseline: phase x wickets'), ('Reg: ridge','Baseline: phase x wickets'), ('Clf: logistic','Baseline: phase x wickets')]:
    ci(se(a,'runs',yr) - se(b,'runs',yr), f'sq error runs  {a} - {b}')
    if 'Reg' not in a + b: ci(ll(a) - ll(b), f'log loss       {a} - {b}')
for m in ('Clf: LightGBM','Clf: logistic'):
    P = cat(m,'prob')
    for nm, p, y in [('wicket', P[:,0], yc[va]==0), ('4 or 5+', P[:,5:].sum(1), yc[va]>=5)]:
        q = pd.qcut(p, 5, labels=False); print(m, nm, 'predicted', [round(p[q==k].mean(),3) for k in range(5)], 'actual', [round(y[q==k].mean(),3) for k in range(5)])

negative = first model better
sq error runs  Clf: LightGBM - Reg: LightGBM: -0.00048  95% [-0.00477, +0.00405]
sq error runs  Clf: logistic - Reg: ridge: +0.00832  95% [+0.00347, +0.01346]
sq error runs  Clf: random forest - Reg: random forest: -0.00527  95% [-0.01036, -0.00020]
sq error runs  Clf: LightGBM - Clf: logistic: -0.02610  95% [-0.03541, -0.01749]


log loss       Clf: LightGBM - Clf: logistic: -0.00893  95% [-0.01192, -0.00613]
sq error runs  Reg: LightGBM - Reg: ridge: -0.01730  95% [-0.02442, -0.01021]
sq error runs  Clf: LightGBM - Baseline: phase x wickets: -0.05255  95% [-0.06262, -0.04236]
log loss       Clf: LightGBM - Baseline: phase x wickets: -0.01405  95% [-0.01678, -0.01117]
sq error runs  Reg: ridge - Baseline: phase x wickets: -0.03477  95% [-0.04567, -0.02345]


sq error runs  Clf: logistic - Baseline: phase x wickets: -0.02645  95% [-0.03951, -0.01244]
log loss       Clf: logistic - Baseline: phase x wickets: -0.00512  95% [-0.00928, -0.00095]
Clf: LightGBM wicket predicted [np.float64(0.041), np.float64(0.051), np.float64(0.059), np.float64(0.07), np.float64(0.101)] actual [np.float64(0.039), np.float64(0.051), np.float64(0.062), np.float64(0.06), np.float64(0.099)]
Clf: LightGBM 4 or 5+ predicted [np.float64(0.107), np.float64(0.145), np.float64(0.171), np.float64(0.202), np.float64(0.249)] actual [np.float64(0.099), np.float64(0.149), np.float64(0.164), np.float64(0.212), np.float64(0.252)]
Clf: logistic wicket predicted [np.float64(0.033), np.float64(0.045), np.float64(0.055), np.float64(0.069), np.float64(0.107)] actual [np.float64(0.042), np.float64(0.053), np.float64(0.056), np.float64(0.066), np.float64(0.094)]
Clf: logistic 4 or 5+ predicted [np.float64(0.103), np.float64(0.148), np.float64(0.184), np.float64(0.225), np.float64(0.294